In [1]:
from xml.etree import ElementTree
import requests
from bs4 import BeautifulSoup
import re
from datetime import datetime

In [2]:
BASE_URL = "https://www.justetf.com/en/etf-profile.html"
DEFAULT_HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

In [3]:
def get_soup(isin:str):
    url = f"{BASE_URL}?isin={isin}"
    session = requests.Session()
    response = session.get(url, headers=DEFAULT_HEADERS)
    if response.status_code != 200:
        raise RuntimeError(
            f"Failed to fetch ETF page for {isin}: status {response.status_code}"
        )
    return BeautifulSoup(response.text, "html.parser")

soup = get_soup("IE00B4L5Y983")
soup

<!DOCTYPE html>
 <html data-testid="translated-page_html" lang="en-XE" prefix="og: http://ogp.me/ns#" xml:lang="en-XE" xmlns="http://www.w3.org/1999/xhtml"> <head> <script>
/*<![CDATA[*/

        window.GLOBALS = window.GLOBALS || {}
    
/*]]>*/
</script> <script src="https://web.cmp.usercentrics.eu/tcf/stub.js"></script>
<script async="" data-settings-id="--DQiTMqEc3oVX" id="usercentrics-cmp" src="https://web.cmp.usercentrics.eu/ui/loader.js"></script>
<script>
 (function() {
   window.addEventListener('UC_UI_INITIALIZED', function(event) {
     if (window.__ucCmp && typeof window.__ucCmp.changeLanguage === 'function') {
         window.__ucCmp.changeLanguage("en").then(function() {
               console.log('Changed Usercentrics language to',"en");
             })['catch'](function(err) {
               console.error('Usercentrics language change failed', err);
             });
         }
   });
  })();
</script>
<script>console.log('Using UserCentrics CMP for consent tracking. Dra

In [4]:
# BASIC DATA
basic_data = {}
basics_table = soup.find("table", attrs={"data-testid": "etf-basics_data_table"})
# Für jeded <tr>:
# class vallable: class val
rows = basics_table.find_all("tr")
for row in rows:
    #print(row)
    label = row.find("td",class_="vallabel").get_text(strip=True)
    # Value in 2nd col
    cols = row.find_all("td")
    value = cols[-1].get_text(strip=True)
    #value = row.find("td",class_=re.compile(r"val\d*")).get_text(strip=True)
    basic_data[label] = value

basic_data

{'Index': 'MSCI World',
 'Investment focus': 'Equity, World',
 'Fund size': 'EUR 132,135 m',
 'Total expense ratio': '0.20% p.a.',
 'Replication': 'Physical(Optimized sampling)',
 'Legal structure': 'ETF',
 'Investment approach': 'Long-only',
 'Sustainability': 'No',
 'Fund currency': 'USD',
 'Currency risk': 'Currency unhedged',
 'Volatility 1 year (in EUR)': '10.54%',
 'Inception/ Listing Date': '25 September 2009',
 'Distribution policy': 'Accumulating',
 'Distribution frequency': '-',
 'Fund domicile': 'Ireland',
 'Fund Provider': 'iShares'}

In [5]:
# TOP 10 HOLDINGS
top_holdings = []
holdings_rows = soup.find_all("tr", attrs={"data-testid": "etf-holdings_top-holdings_row"})
for row in holdings_rows:
    name_elem = row.find("a", attrs={"data-testid": "tl_etf-holdings_top-holdings_link_name"})
    pct_elem = row.find("span",attrs={"data-testid": "tl_etf-holdings_top-holdings_value_percentage"},)
    if name_elem and pct_elem:
        holding_name = name_elem.get_text(strip=True)
        pct = pct_elem.get_text(strip=True)
        top_holdings.append((holding_name, pct))
top_holdings

[('NVIDIA Corp.', '5.53%'),
 ('Apple', '5.07%'),
 ('Microsoft Corp', '3.90%'),
 ('Amazon.com, Inc.', '2.74%'),
 ('Alphabet, Inc. A', '2.17%'),
 ('Broadcom Inc.', '1.82%'),
 ('Alphabet, Inc. C', '1.71%'),
 ('Meta Platforms', '1.37%'),
 ('Micron Technology', '1.18%'),
 ('Tesla', '1.15%')]

In [6]:
def _fetch_ajax_data_new(session: requests.Session, counter:int, isin: str, endpoint: str) -> Optional[str]:
    """
    Fetch expanded data via AJAX request (for countries/sectors load more).

    Args:
        session: requests Session with cookies from main page
        isin: The ISIN of the ETF
        endpoint: The AJAX endpoint path (e.g., 'holdingsSection-countries-loadMoreCountries')
        element_id: The element ID for Wicket (e.g., 'id6')

    Returns:
        HTML content from AJAX response, or None if failed
    """

    #timestamp = int(datetime.timestamp(datetime.now())*1000)
    ajax_url =f"{BASE_URL}?{counter}-1.0-{endpoint}&isin={isin}&_wicket=1"#&_={timestamp}"

    ajax_headers = {
        **DEFAULT_HEADERS,
        "X-Requested-With": "XMLHttpRequest",
        "Wicket-Ajax": "true",
        "Wicket-Ajax-Baseurl": f"en/etf-profile.html?isin={isin}",
        "Accept": "application/xml, text/xml, */*; q=0.01",
        "Referer": f"{BASE_URL}?isin={isin}",
    }

    try:
        resp = session.get(ajax_url, headers=ajax_headers)
        print(resp)
        if resp.status_code == 200 and resp.text:
            print("Ajax call OK")
            return resp.text
    except Exception as e:
        print(f"AJAX request error: {e}")

    return None

In [27]:
# Try AJAX for full list
session = requests.Session()
isin = "IE00B4L5Y983"
isin = "IE00B5BMR087"
counter = 0

# Establish cookies/session state first
main_response = session.get(
    f"{BASE_URL}?isin={isin}",
    headers=DEFAULT_HEADERS,
)
main_response.raise_for_status()

countries_xml = _fetch_ajax_data_new(
    session,
    counter,
    isin,
    "holdingsSection-countries-loadMoreCountries",
)

<Response [200]>
Ajax call OK


In [28]:
print(countries_xml)

<ajax-response><redirect><![CDATA[/en/access-denied.html]]></redirect></ajax-response>


In [29]:
root = ElementTree.fromstring(countries_xml)
components = root.findall("component")
bool(components)

False

In [13]:
def parse_allocation_from_ajax_new(xml_response: str, table_testid: str, name_testid: str, pct_testid: str) -> List[AllocationItem]:

    allocations = []

    try:
        root = ElementTree.fromstring(xml_response)
        components = root.findall("component")
        for component in components:
            soup = BeautifulSoup(component.text, "html.parser")
            table = soup.find("table",attrs={"data-testid":table_testid})
            if table:
                rows = soup.find_all("tr", attrs={"data-testid": True})
                for row in rows:
                    name_elem = row.find("td", attrs={"data-testid": name_testid})
                    pct_elem = row.find("span", attrs={"data-testid": pct_testid})
                    if name_elem and pct_elem:
                        name = name_elem.get_text(strip=True)
                        #pct = _parse_percentage(pct_elem.get_text(strip=True))
                        pct = pct_elem.get_text(strip=True)
                        if name and pct is not None:
                            allocations.append(
                                (name, pct)
                                #AllocationItem(name=name, percentage=pct)
                            )
    except Exception as e:
        print(f"Error parsing allocation data: {e}")

    return allocations

parse_allocation_from_ajax_new(countries_xml, 
                               "etf-holdings_countries_table", 
                               "tl_etf-holdings_countries_value_name", 
                               "tl_etf-holdings_countries_value_percentage")


[]

In [14]:
root = ElementTree.fromstring(countries_xml)
components = root.findall("component")
for component in components:
    soup = BeautifulSoup(component.text, "html.parser")
    table = soup.find("table",attrs={"data-testid":"etf-holdings_countries_table"})
    if table:
        rows = soup.find_all("tr", attrs={"data-testid": True})
        for row in rows:
            name_elem = row.find("td",attrs={"data-testid":"tl_etf-holdings_countries_value_name"})
            name = name_elem.get_text(strip=True)
            pct_elem = row.find("span",attrs={"data-testid":"tl_etf-holdings_countries_value_percentage"})
            pct = pct_elem.get_text(strip=True)
            print(name, pct)
